# Problem 3: Multilingual Retrieval Augmented Generation (25 points)

Implement a multilingual search and retrieval augmented generation system using the OPUS Books dataset, which contains parallel text in English and Italian. You will create a system that can search across languages and generate content based on the retrieved passages.

## Problem 3(a): Setting up the vector search system (8 points)

- Use sentence-transformers' multilingual model `paraphrase-multilingual-MiniLM-L12-v2`
- Create vector embeddings for the OPUS Books text passages
- Build a FAISS index for efficient similarity search
- Save and load the index for reuse

In [ ]:
# ── Required packages (Python 3.12.6, tested versions) ─────────────────────
# Pinned install (recommended for reproducibility):
# !pip install \
#     torch==2.10.0 \
#     numpy==2.4.2 \
#     faiss-cpu==1.13.2 \
#     sentence-transformers==5.2.3 \
#     transformers==5.2.0 \
#     datasets==4.5.0 \
#     tqdm==4.67.2 \
#     sentencepiece==0.2.1 \
#     accelerate==1.12.0
#
# Latest versions (may need minor adjustments):
!pip install torch numpy faiss-cpu sentence-transformers transformers datasets tqdm sentencepiece accelerate

import os

# macOS: prevent Metal (MPS) memory conflicts when loading multiple large models
os.environ.setdefault("PYTORCH_MPS_HIGH_WATERMARK_RATIO", "0.0")

import numpy as np
import torch
import faiss
import time
import json
from tqdm import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import MBartForConditionalGeneration, MBart50Tokenizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 93.2 MB/s eta 0:00:00


#### Loading and Processing the Dataset

In [ ]:
# Load 1000 parallel English-Italian text pairs from OPUS Books
dataset = load_dataset("opus_books", "en-it", split="train[:1000]")

texts_en = [item['translation']['en'] for item in dataset]
texts_it = [item['translation']['it'] for item in dataset]

print(f"Loaded {len(texts_en)} English passages and {len(texts_it)} Italian passages.")
print(f"\nExample English : {texts_en[0][:100]}...")
print(f"Example Italian : {texts_it[0][:100]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en-it/train-00000-of-00001.parquet:   0%|          | 0.00/5.73M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/32332 [00:00<?, ? examples/s]

Loaded 1000 English passages and 1000 Italian passages.

Example English : Source: Project Gutenberg...
Example Italian : Source: www.liberliber.it/Audiobook available here...


#### Initialization

In [ ]:
# Initialize the multilingual sentence transformer for creating embeddings
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device='cpu')

# Initialize the mBART model and tokenizer for multilingual text generation
generator_tokenizer = MBart50Tokenizer.from_pretrained(
    'facebook/mbart-large-50-many-to-many-mmt'
)
generator_model = MBartForConditionalGeneration.from_pretrained(
    'facebook/mbart-large-50-many-to-many-mmt',
    torch_dtype=torch.float32,
    device_map="cpu",
)

print(f"Sentence transformer loaded: {model.get_sentence_embedding_dimension()}-dim embeddings")
print(f"mBART model and tokenizer loaded.")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Sentence transformer loaded: 384-dim embeddings
mBART model and tokenizer loaded.


#### Create Embeddings

In [ ]:
# Generate embeddings for all English and Italian passages, then combine them.
embeddings_en = model.encode(texts_en, show_progress_bar=True, convert_to_numpy=True)
embeddings_it = model.encode(texts_it, show_progress_bar=True, convert_to_numpy=True)
embeddings    = np.vstack([embeddings_en, embeddings_it]).astype(np.float32)

print(f"Embedding shape: {embeddings.shape}")


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Embedding shape: (2000, 384)


#### FAISS Indexing for Efficient Similarity Search

In [ ]:
def build_faiss_index(embeddings: np.ndarray) -> faiss.IndexFlatL2:
    """
    Build a FAISS flat L2 index from the given embedding matrix.

    Parameters
    ----------
    embeddings : np.ndarray, shape (n, d) -- should be float32

    Returns
    -------
    faiss.IndexFlatL2
    """
    d = embeddings.shape[1]
    index = faiss.IndexFlatL2(d)
    index.add(embeddings.astype(np.float32))
    return index


index = build_faiss_index(embeddings)
print(f"Index contains {index.ntotal} vectors.")

# Save the index to disk and reload it
faiss.write_index(index, "multilingual_index.faiss")
index = faiss.read_index("multilingual_index.faiss")
print("Index saved and reloaded successfully.")


Index contains 2000 vectors.
Index saved and reloaded successfully.


## Problem 3(b): Implement Multilingual Search (8 points)

- Create a search function that accepts queries in either English or Italian
- Add metadata filtering capability to search in specific languages
- Return top-k most relevant passages with scores
- Implement efficient batch processing for multiple queries

#### Implementing Multilingual Search

In [ ]:
def search(query: str, lang: str = None, k: int = 5) -> list:
    """
    Perform semantic search over the multilingual FAISS index.

    Parameters
    ----------
    query : str       -- query text in English or Italian
    lang  : str|None  -- restrict results to 'en', 'it', or None for both
    k     : int       -- number of results to return

    Returns
    -------
    list[dict] -- each dict has keys 'text' (str), 'lang' (str), 'score' (float)
    """
    # Encode the query
    query_vector = model.encode([query]).astype(np.float32)

    # Search more than k so we still have enough after language filtering
    fetch_k = k * 4 if lang else k
    D, I = index.search(query_vector, fetch_k)

    results = []
    n_en = len(texts_en)
    for dist, idx in zip(D[0], I[0]):
        if idx < 0:
            continue
        if idx < n_en:
            text = texts_en[idx]
            lang_code = 'en'
        else:
            text = texts_it[idx - n_en]
            lang_code = 'it'

        # Apply language filter
        if lang is not None and lang_code != lang:
            continue

        results.append({
            'text': text,
            'lang': lang_code,
            'score': float(dist),
        })

        if len(results) >= k:
            break

    return results


#### Testing Multilingual Search

In [ ]:
# Test queries in both languages
queries = {
    "English": "stories about adventure and discovery",
    "Italian": "storie di avventura e scoperta"
}

for lang_name, query_text in queries.items():
    print(f"\n{'='*60}")
    print(f"{lang_name} query: '{query_text}'")
    print('='*60)

    start = time.time()
    results = search(query_text, k=3)
    elapsed = time.time() - start

    print(f"Search time: {elapsed*1000:.2f} ms  |  {len(results)} result(s)")
    for i, r in enumerate(results, 1):
        print(f"  [{i}] ({r['lang'].upper()}, score={r['score']:.4f})  {r['text'][:90]}...")

# Cross-lingual: English query filtered to Italian results only
print(f"\n{'='*60}")
print("English query -> Italian results only")
print('='*60)
results_it = search("stories about adventure and discovery", lang='it', k=3)
for i, r in enumerate(results_it, 1):
    print(f"  [{i}] ({r['lang'].upper()}, score={r['score']:.4f})  {r['text'][:90]}...")


English query: 'stories about adventure and discovery'
Search time: 46.85 ms  |  3 result(s)
  [1] (IT, score=20.6937)  Ogni incisione mi narrava una storia, spesso misteriosa per la mia intelligenza poco svilu...
  [2] (EN, score=20.8832)  I considered it a narrative of facts, and discovered in it a vein of interest deeper than ...
  [3] (EN, score=21.6437)  Each picture told a story; mysterious often to my undeveloped understanding and imperfect ...

Italian query: 'storie di avventura e scoperta'
Search time: 42.04 ms  |  3 result(s)
  [1] (IT, score=14.7477)  Ogni incisione mi narrava una storia, spesso misteriosa per la mia intelligenza poco svilu...
  [2] (EN, score=15.2384)  I considered it a narrative of facts, and discovered in it a vein of interest deeper than ...
  [3] (IT, score=15.3510)  — Abbastanza....

English query -> Italian results only
  [1] (IT, score=20.6937)  Ogni incisione mi narrava una storia, spesso misteriosa per la mia intelligenza poco svilu...
  [2] (IT,

## Problem 3(c): Adding Retrieval-Augmented Generation (RAG) Capabilities (9 points)

In this section you will add RAG functionality by combining the search system above with
the mBART-large-50 language model.

### Tasks

1. **Content generation** -- implement `generate_content(prompt, context)` that feeds a
   retrieved passage plus an instruction prompt into the mBART model and returns the
   generated text.

2. **Single-document RAG** -- implement `rag_single(query, prompt)` that retrieves the
   single most relevant passage and generates content based on it.

3. **Multi-document RAG** -- implement `rag_group(query, prompt, k)` that retrieves the
   top-k passages, concatenates them as context, and generates a comparative response.

4. **Prompt strategies** -- experiment with different prompt types:
   - *Recommendation prompt*: `"Write a short book recommendation based on this excerpt:"`
   - *Comparative prompt*: `"Compare and contrast these excerpts, discussing themes and style:"`

5. **Testing** -- run the functions with:
   - English query: `"stories about adventure and discovery"`
   - Italian query:  `"storie di avventura e scoperta"`

#### Implementing RAG Capabilities

In [ ]:
def generate_content(prompt: str, context: str) -> str:
    """
    Generate text using the mBART model conditioned on a prompt and context.

    Parameters
    ----------
    prompt  : str -- instruction for the model
    context : str -- retrieved passage(s) to base the generation on

    Returns
    -------
    str -- generated text
    """
    input_text = f"{prompt}\n{context}"
    inputs = generator_tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    )
    outputs = generator_model.generate(
        **inputs,
        forced_bos_token_id=generator_tokenizer.lang_code_to_id["en_XX"],
        max_new_tokens=200,
    )
    return generator_tokenizer.decode(outputs[0], skip_special_tokens=True)


def rag_single(query: str, prompt: str) -> str:
    """
    Retrieve the single most relevant passage and generate content based on it.

    Parameters
    ----------
    query  : str -- search query
    prompt : str -- generation instruction

    Returns
    -------
    str -- generated text
    """
    results = search(query, k=1)
    context = results[0]['text']
    return generate_content(prompt, context)


def rag_group(query: str, prompt: str, k: int = 3) -> str:
    """
    Retrieve the top-k passages and generate content based on all of them.

    Parameters
    ----------
    query  : str -- search query
    prompt : str -- generation instruction
    k      : int -- number of passages to retrieve

    Returns
    -------
    str -- generated text
    """
    results = search(query, k=k)
    context = "\n---\n".join([r['text'] for r in results])
    return generate_content(prompt, context)


#### Testing RAG Capabilities

In [ ]:
# Single-document RAG
print("=" * 60)
print("Single-Document RAG")
print("=" * 60)
result = rag_single(
    query="stories about adventure and discovery",
    prompt="Write a short book recommendation based on this excerpt:"
)
print(result)

# Multi-document RAG (English query)
print("\n" + "=" * 60)
print("Multi-Document RAG  (English query)")
print("=" * 60)
result = rag_group(
    query="stories about adventure and discovery",
    prompt="Compare and contrast these book excerpts, discussing their themes and style:",
    k=3
)
print(result)

# Multi-document RAG (Italian query)
print("\n" + "=" * 60)
print("Multi-Document RAG  (Italian query)")
print("=" * 60)
result = rag_group(
    query="storie di avventura e scoperta",
    prompt="Confronta e analizza questi estratti di libri, discutendo i temi e lo stile:",
    k=3
)
print(result)

Single-Document RAG
Schreiben Sie eine kurze Buchrecommendation basierend auf diesem Auszug: Ogni incision mi narrava una storia, spesso misteriosa per la mia intelligenza poco sviluppata e per il mio incompleto sentimento, ma sempre interessantissima; così interessante come i racconti che ci faceva Bessie nelle serate invernali quando era di buon umore e quando, dopo aver portato la tavola da stirare nella stanza dei bambini, ci permetteva di sedersi vicino a lei. Allora, pieghettando le sciarpe di trina della signora Reed e le cuffie da notte, ci riscaldava la fantasia con narrazioni di amore e di avventure, tolte dai vecchi racconti di fate e dalle antiche ballate, o, come mi accorsi più tardi, da Pamela e da Enrico, conte di Mareland.

Multi-Document RAG  (English query)
Ogni incisione mi narrava una storia, spesso misteriosa per la mia intelligenza poco sviluppata e per il mio incompleto sentimento, ma sempre interessantissima; così interessante come i racconti che ci faceva Bessi

## Bonus (+5 points): Semantic Caching

Implement a `SemanticCache` class to avoid redundant searches for repeated or highly
similar queries.

- Store each query's embedding alongside its results.
- For a new query, compute cosine similarity against all cached embeddings.
- If the maximum similarity exceeds a threshold (e.g. 0.95), return the cached results.
- Otherwise run a fresh search and add the result to the cache.

Measure and report the cache hit rate and average query latency with and without caching.

In [ ]:
class SemanticCache:
    def __init__(self, threshold: float = 0.95):
        """
        Parameters
        ----------
        threshold : float -- cosine similarity above which a cache hit is declared
        """
        self.threshold = threshold
        self.cache = []  # list of (query_embedding np.ndarray, results list)

    def _cosine_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        """Compute cosine similarity between two 1-D vectors."""
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

    def search_with_cache(self, query: str, lang: str = None, k: int = 5) -> list:
        """
        Return cached results for semantically similar past queries;
        otherwise run a fresh search and cache the result.

        Parameters
        ----------
        query : str
        lang  : str|None
        k     : int

        Returns
        -------
        list[dict] -- same format as search()
        """
        query_emb = model.encode([query]).astype(np.float32)[0]

        # Check cache for a hit
        for cached_emb, cached_results in self.cache:
            sim = self._cosine_similarity(query_emb, cached_emb)
            if sim >= self.threshold:
                return cached_results  # cache hit

        # Cache miss: run fresh search
        results = search(query, lang=lang, k=k)
        self.cache.append((query_emb, results))
        return results


# ── Test semantic caching ────────────────────────────────────────────────────
cache = SemanticCache(threshold=0.95)
test_queries = [
    "stories about adventure and discovery",
    "tales of adventure and exploration",        # semantically very similar
    "storie di avventura e scoperta",            # Italian, different embedding
    "stories about adventure and discovery",     # exact repeat
    "tales about journeys and finding new lands", # similar but may be below threshold
]

n_hits = 0
n_total = len(test_queries)
latencies_cached = []
latencies_uncached = []

print("Semantic Cache Test")
print("=" * 60)
for q in test_queries:
    cache_size_before = len(cache.cache)
    t0 = time.time()
    results = cache.search_with_cache(q, k=3)
    elapsed = (time.time() - t0) * 1000
    cache_size_after = len(cache.cache)
    is_hit = (cache_size_after == cache_size_before)

    if is_hit:
        n_hits += 1
        latencies_cached.append(elapsed)
    else:
        latencies_uncached.append(elapsed)

    print(f"  Query: {q[:50]:50s} | {elapsed:6.2f} ms | {'HIT' if is_hit else 'MISS'}")
    for r in results[:2]:
        print(f"    -> ({r['lang'].upper()}) {r['text'][:70]}...")

print(f"\nCache entries: {len(cache.cache)}")
print(f"Hit rate: {n_hits}/{n_total} = {n_hits/n_total:.0%}")
if latencies_cached:
    print(f"Avg latency (cache hit):  {np.mean(latencies_cached):.2f} ms")
if latencies_uncached:
    print(f"Avg latency (cache miss): {np.mean(latencies_uncached):.2f} ms")
if latencies_cached and latencies_uncached:
    print(f"Speedup from caching: {np.mean(latencies_uncached)/np.mean(latencies_cached):.1f}x")

Semantic Cache Test
  Query: stories about adventure and discovery              |  68.65 ms | MISS
    -> (IT) Ogni incisione mi narrava una storia, spesso misteriosa per la mia int...
    -> (EN) I considered it a narrative of facts, and discovered in it a vein of i...
  Query: tales of adventure and exploration                 |  57.12 ms | MISS
    -> (EN) I considered it a narrative of facts, and discovered in it a vein of i...
    -> (IT) Ogni incisione mi narrava una storia, spesso misteriosa per la mia int...
  Query: storie di avventura e scoperta                     |  28.83 ms | HIT
    -> (IT) Ogni incisione mi narrava una storia, spesso misteriosa per la mia int...
    -> (EN) I considered it a narrative of facts, and discovered in it a vein of i...
  Query: stories about adventure and discovery              |  27.78 ms | HIT
    -> (IT) Ogni incisione mi narrava una storia, spesso misteriosa per la mia int...
    -> (EN) I considered it a narrative of facts, and discovered